# Lab: Bag of Words and TF-IDF
**Course:** Natural Language Processing  |  **Topic:** Converting text to numbers

In earlier labs we cleaned text with NLTK (tokenization, stop words, stemming). Today we turn the cleaned text into numbers that a machine learning model can use.

**Flow of this lab:** raw text, then clean text, then Bag of Words matrix, then TF-IDF matrix, then compare documents.

Part 1 to Part 9 is a demo. Run each cell with Shift + Enter and read the explanation below it. The complete task for you to solve is at the end of the notebook.

## Part 1: Setup

In [ ]:
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("Setup done")

## Part 2: The dataset
Eight short SMS messages: 4 spam and 4 normal (ham). The data is small so that you can read every row.

In [ ]:
messages = [
    "WINNER!! You have won a free prize. Call now to claim your reward",
    "Free entry in our weekly contest. Text WIN to 80085 now",
    "URGENT! Your account is locked. Verify now to avoid suspension",
    "Get a free ringtone. Reply YES to claim your free offer today",
    "Are we still meeting for lunch tomorrow?",
    "I will call you when I reach home",
    "Can you send me the notes from today's class?",
    "Don't forget to bring your laptop to the lab",
]
labels = ["spam", "spam", "spam", "spam", "ham", "ham", "ham", "ham"]

df = pd.DataFrame({"text": messages, "label": labels})
df

## Part 3: Cleaning the text (what we did before)
We apply the same steps as in the earlier labs: tokenize, lowercase, remove punctuation, remove stop words, stem. First on one message, step by step.

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

msg = df.text[0]
print("Original   :", msg)

tokens = word_tokenize(msg.lower())
print("Tokens     :", tokens)

words = [t for t in tokens if t.isalpha()]
print("Words only :", words)

no_stop = [t for t in words if t not in stop_words]
print("No stopword:", no_stop)

stems = [stemmer.stem(t) for t in no_stop]
print("Stemmed    :", stems)

Now we put the same steps into one function, so we can use it on every message.

In [ ]:
def clean_tokens(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return [stemmer.stem(t) for t in tokens]

def clean_text(text):
    return " ".join(clean_tokens(text))

df["clean"] = df.text.apply(clean_text)
df[["text", "clean"]]

## Part 4: Bag of Words with CountVectorizer
Idea: put the words of a document in a "bag". We ignore the order and only count how many times each word appears.

Steps: (1) build a vocabulary of all unique words, (2) count each word in each document. We first try it on three tiny sentences.

In [ ]:
docs = ["the cat sat on the mat",
        "the dog sat on the log",
        "the cat chased the dog"]

cv = CountVectorizer()
X = cv.fit_transform(docs)

print("Vocabulary (word: column number):")
print(cv.vocabulary_)
print()
print("Column names:", list(cv.get_feature_names_out()))

In [ ]:
bow = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=["D1", "D2", "D3"])
bow

How to read the table: each row is one document, each column is one word, and each number is how many times that word appears in the document. The word "the" appears twice in every document.

`fit_transform` does two things: it learns the vocabulary (fit) and then counts the words (transform).

## Part 5: Bag of Words on our messages
Now we use the cleaned messages from Part 3.

In [ ]:
cv_msg = CountVectorizer()
X_msg = cv_msg.fit_transform(df.clean)

print("Matrix shape (messages x words):", X_msg.shape)

bow_msg = pd.DataFrame(X_msg.toarray(), columns=cv_msg.get_feature_names_out())
bow_msg

Most cells are zero. This is normal: each message uses only a few words out of the whole vocabulary. A matrix like this is called a sparse matrix.

## Part 6: Feature engineering (choosing the columns)
`CountVectorizer` has options that change the vocabulary. The table shows how the vocabulary size changes with each option on the raw messages.

In [ ]:
options = {
    "default":                 CountVectorizer(),
    "stop_words='english'":    CountVectorizer(stop_words="english"),
    "ngram_range=(1, 2)":      CountVectorizer(ngram_range=(1, 2)),
    "min_df=2":                CountVectorizer(min_df=2),
}

rows = []
for name, vec in options.items():
    Xo = vec.fit_transform(df.text)
    rows.append((name, Xo.shape[1]))

pd.DataFrame(rows, columns=["option", "vocabulary size"])

- `stop_words='english'` removes common words, so the vocabulary gets smaller.
- `ngram_range=(1, 2)` also adds word pairs such as "free prize", so the vocabulary gets bigger.
- `min_df=2` keeps only words that appear in at least 2 messages, so the vocabulary gets much smaller.

## Part 7: The problem with counts
Let us see which words appear most often in our cleaned messages.

In [ ]:
totals = pd.Series(X_msg.toarray().sum(axis=0), index=cv_msg.get_feature_names_out())
totals.sort_values(ascending=False).head(8)

Words that are frequent are not always the most useful words. A word that appears in every message does not help us tell one message from another. We want a score that gives a high weight to words that are special to a document. This is TF-IDF.

## Part 8: TF-IDF
TF-IDF = TF x IDF

- **TF (term frequency):** how often the word appears in this document.
- **IDF (inverse document frequency):** how rare the word is across all documents. A word in every document gets a low IDF. A word in only one document gets a high IDF.

Textbook formulas: TF = count / number of words in the document, and IDF = log10(N / df), where N is the number of documents and df is the number of documents that contain the word.

Example with our 3 sentences (N = 3): the word "the" is in all 3 documents, so IDF = log10(3/3) = 0. The word "chased" is in only 1 document, so IDF = log10(3/1) = 0.477. So "the" gets weight 0 and "chased" gets a high weight.

Now the same idea with `TfidfVectorizer`.

In [ ]:
tv = TfidfVectorizer()
T = tv.fit_transform(docs)

tfidf = pd.DataFrame(T.toarray(), columns=tv.get_feature_names_out(), index=["D1", "D2", "D3"])
tfidf.round(2)

In [ ]:
idf_values = pd.Series(tv.idf_, index=tv.get_feature_names_out())
idf_values.round(2).sort_values()

The words with the lowest IDF ("the") appear in many documents. The words with the highest IDF ("chased", "log", "mat") appear in only one document.

Note: scikit-learn uses a slightly different formula from the textbook (it smooths the IDF and normalizes each row), so the numbers are not exactly the same as the hand calculation, but the idea is the same.

## Part 9: BoW and TF-IDF on our messages, and document similarity
We compare the top 3 words of a message under BoW and under TF-IDF.

In [ ]:
cv_s = CountVectorizer(analyzer=clean_tokens)
Xc = cv_s.fit_transform(df.text)

tv_s = TfidfVectorizer(analyzer=clean_tokens)
Xt = tv_s.fit_transform(df.text)

names = tv_s.get_feature_names_out()

def top_words(row, k=3):
    values = row.toarray().ravel()
    best = values.argsort()[::-1][:k]
    return [(names[i], round(float(values[i]), 2)) for i in best]

for i in [0, 3, 6]:
    print(df.text[i])
    print("   BoW top 3   :", top_words(Xc[i]))
    print("   TF-IDF top 3:", top_words(Xt[i]))
    print()

Cosine similarity tells us how similar two documents are. It is 1 for identical documents and 0 for documents that share no words.

In [ ]:
sim = cosine_similarity(Xt)
pd.DataFrame(sim.round(2))

In [ ]:
sim_copy = sim.copy()
np.fill_diagonal(sim_copy, -1)          # ignore a message compared with itself
i, j = np.unravel_index(sim_copy.argmax(), sim_copy.shape)

print("Most similar pair:")
print(df.text[i])
print(df.text[j])
print("Similarity:", round(sim_copy[i, j], 2))

## Summary
- Computers need numbers, so we convert each document into a vector.
- Bag of Words: build a vocabulary and count the words. Word order is ignored.
- `CountVectorizer` builds the BoW matrix. Its options (stop words, n-grams, min_df) change the columns.
- TF-IDF gives a high weight to words that are special to a document and a low weight to words found everywhere.
- Cosine similarity compares two document vectors.
- Cleaning the text first (earlier labs) gives a smaller and better vocabulary.

---
# Complete Task (to be solved by the student)

You are given 10 short product reviews: 5 positive and 5 negative. Using the code from the demo above, complete all the steps below. Write your code in the empty cells and your answers in the text cells.

Run the next cell first to load the data.

In [ ]:
reviews = [
    "I love this phone, the battery lasts all day",
    "Great camera and very fast delivery",
    "The screen is bright and the sound is great",
    "Excellent quality, I love it",
    "Very happy with this phone, works great",
    "The battery is terrible and drains fast",
    "Poor quality, the screen broke in a week",
    "Delivery was slow and the box was damaged",
    "I hate this phone, the camera is bad",
    "Terrible sound and very slow performance",
]
review_labels = ["positive"] * 5 + ["negative"] * 5

task_df = pd.DataFrame({"text": reviews, "label": review_labels})
task_df

### Step 1: Clean the reviews
Apply the `clean_text` function from Part 3 to every review and store the result in a new column called `clean`. Display the `text` and `clean` columns.

In [ ]:
# Write your code here

### Step 2: Bag of Words
Use `CountVectorizer` on the cleaned reviews. Print the vocabulary, print the shape of the matrix, and display the matrix as a DataFrame with the words as column names.

In [ ]:
# Write your code here

### Step 3: Feature options
Build a table of vocabulary sizes for the raw review text using three settings: the default, `stop_words='english'`, and `ngram_range=(1, 2)`.

In [ ]:
# Write your code here

### Step 4: TF-IDF
Use `TfidfVectorizer` on the cleaned reviews. Display the TF-IDF matrix as a DataFrame rounded to 2 decimals. Then print the 5 words with the lowest IDF and the 5 words with the highest IDF.

In [ ]:
# Write your code here

### Step 5: Compare BoW and TF-IDF
For review 0, review 5 and review 8, print the top 3 words under BoW and under TF-IDF (use the same idea as `top_words` in Part 9).

In [ ]:
# Write your code here

### Step 6: Similarity
Calculate the cosine similarity between all reviews using the TF-IDF matrix. Find and print the most similar pair of reviews and their similarity score.

In [ ]:
# Write your code here

### Step 7: Submission
Run the notebook from top to bottom (Runtime, Restart and run all) and submit it with all outputs and your answers.